In [1]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

#DATA_PATH = Path("Data/football.csv")  # relative path so it runs on any machine

# ## Step 1: Load and inspect

In [2]:
df = pd.read_csv('C:/Users/cgrogan4/PycharmProjects/Football/Data/football.csv')
print("rank_away missing:", df["rank_away"].isna().mean())
print(df.shape)
df.info()  # note: call it, don't print the method
print(df.head())

rank_away missing: 0.8217507433351812
(19843, 58)
<class 'pandas.DataFrame'>
RangeIndex: 19843 entries, 0 to 19842
Data columns (total 58 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   season                 19843 non-null  int64  
 1   week                   18978 non-null  float64
 2   date                   19843 non-null  str    
 3   time_et                19843 non-null  str    
 4   game_type              19843 non-null  str    
 5   away                   19843 non-null  str    
 6   home                   19843 non-null  str    
 7   rank_away              3537 non-null   float64
 8   rank_home              4425 non-null   float64
 9   conf_away              17576 non-null  str    
 10  conf_home              19832 non-null  str    
 11  neutral                19843 non-null  bool   
 12  score_away             19843 non-null  int64  
 13  score_home             19843 non-null  int64  
 14  q1_away        

# ## Step 2: Standardize column names

In [3]:
df.columns = (
    df.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
)
print(df.columns.tolist())

['season', 'week', 'date', 'time_et', 'game_type', 'away', 'home', 'rank_away', 'rank_home', 'conf_away', 'conf_home', 'neutral', 'score_away', 'score_home', 'q1_away', 'q2_away', 'q3_away', 'q4_away', 'ot_away', 'q1_home', 'q2_home', 'q3_home', 'q4_home', 'ot_home', 'first_downs_away', 'first_downs_home', 'third_down_comp_away', 'third_down_att_away', 'third_down_comp_home', 'third_down_att_home', 'fourth_down_comp_away', 'fourth_down_att_away', 'fourth_down_comp_home', 'fourth_down_att_home', 'pass_comp_away', 'pass_att_away', 'pass_yards_away', 'pass_comp_home', 'pass_att_home', 'pass_yards_home', 'rush_att_away', 'rush_yards_away', 'rush_att_home', 'rush_yards_home', 'total_yards_away', 'total_yards_home', 'fum_away', 'fum_home', 'int_away', 'int_home', 'pen_num_away', 'pen_yards_away', 'pen_num_home', 'pen_yards_home', 'possession_away', 'possession_home', 'attendance', 'tv']


# ## Step 3: Remove duplicates

In [4]:
n_before = len(df)
df = df.drop_duplicates()
print(f"Removed {n_before - len(df)} duplicate rows")

Removed 0 duplicate rows


# ## Step 4: Examine missing values (once)


In [5]:
missing_pct = df.isna().mean().mul(100).round(2).sort_values(ascending=False)
print(missing_pct[missing_pct > 0])

rank_away                82.18
rank_home                77.70
tv                       25.26
attendance               21.23
conf_away                11.42
possession_away           8.12
possession_home           7.99
first_downs_away          7.97
third_down_att_away       7.97
third_down_comp_away      7.97
pass_comp_away            7.97
pass_att_away             7.97
pass_yards_away           7.97
rush_yards_away           7.97
total_yards_away          7.97
fum_away                  7.97
int_away                  7.97
rush_att_away             7.97
fourth_down_comp_away     7.97
fourth_down_att_away      7.97
pen_yards_away            7.97
pen_num_away              7.97
pass_yards_home           7.95
pass_comp_home            7.95
first_downs_home          7.95
third_down_att_home       7.95
pass_att_home             7.95
int_home                  7.95
rush_yards_home           7.95
total_yards_home          7.95
rush_att_home             7.95
third_down_comp_home      7.95
fourth_d

# ## Step 5: Build the target
# Rows with no final score have no label, so drop them rather than imputing a score.
# Draws count as 0 (not a home win).

In [6]:
df = df.dropna(subset=["score_home", "score_away"]).copy()
df["home_win"] = (df["score_home"] > df["score_away"]).astype(int)

# ## Step 6: Split BEFORE imputing

In [11]:
LEAKY_COLS = ["score_home", "score_away"]
X = df.drop(columns=LEAKY_COLS + ["home_win"])
y = df["home_win"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=50, stratify=y
)
X_train, X_test = X_train.copy(), X_test.copy()

# Safety check: rank_away should still be ~82% missing here
assert X_train["rank_away"].isna().mean() > 0.5, "rank_away already filled before Step 7"

# Rankings: missing = unranked
for c in ["rank_home", "rank_away"]:
    for d in (X_train, X_test):
        d[c + "_unranked"] = d[c].isna().astype(int)
        d[c] = d[c].fillna(26)

# Text columns: missing is its own category
for c in ["tv", "conf_home", "conf_away"]:
    for d in (X_train, X_test):
        d[c] = d[c].fillna("none")

# Remaining numeric columns: train-set medians
num_cols = X_train.select_dtypes(include="number").columns
medians = X_train[num_cols].median()
X_train[num_cols] = X_train[num_cols].fillna(medians)
X_test[num_cols] = X_test[num_cols].fillna(medians)

print(X_train.isna().sum().sort_values(ascending=False).head(10))
print(X_train["rank_away"].describe())
print(X_train["rank_away_unranked"].mean())

season       0
week         0
date         0
time_et      0
game_type    0
away         0
home         0
rank_away    0
rank_home    0
conf_away    0
dtype: int64
count    15874.000000
mean        23.686846
std          5.808806
min          1.000000
25%         26.000000
50%         26.000000
75%         26.000000
max         26.000000
Name: rank_away, dtype: float64
0.8206501196925791


In [15]:
# Date/time features
for d in (X_train, X_test):
    d["date"] = pd.to_datetime(d["date"])
    d["month"] = d["date"].dt.month
    d["day_of_week"] = d["date"].dt.dayofweek
    d["hour"] = pd.to_datetime(d["time_et"], errors="coerce").dt.hour
    d.drop(columns=["date", "time_et"], inplace=True)

# Categoricals (works whether text is 'object' or 'str')
cat_cols = X_train.select_dtypes(exclude="number").columns
for c in cat_cols:
    cats = pd.CategoricalDtype(X_train[c].unique())
    X_train[c] = X_train[c].astype(cats)
    X_test[c] = X_test[c].astype(cats)

print(cat_cols.tolist())
print(X_train.dtypes.value_counts())

['game_type', 'away', 'home', 'conf_away', 'conf_home', 'neutral', 'tv']
float64     46
int64        3
int32        3
category     2
category     1
category     1
category     1
category     1
category     1
Name: count, dtype: int64


C:\Users\cgrogan4\AppData\Local\Temp\ipykernel_13268\3409491417.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  d["hour"] = pd.to_datetime(d["time_et"], errors="coerce").dt.hour
C:\Users\cgrogan4\AppData\Local\Temp\ipykernel_13268\3409491417.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  d["hour"] = pd.to_datetime(d["time_et"], errors="coerce").dt.hour


In [17]:
df.to_csv('clean_football_data.csv', index=False)